# 03 — Large-footprint waveform simulation

This notebook shows how to simulate large-footprint waveforms from an ALS point cloud and extract relative height (RH) metrics.  Default parameters target **GEDI** (25 m footprint, 0.15 m bins); other instruments (LVIS, GLAS) can be targeted by adjusting `footprint_radius` and `sigma` — see the `waveform` module docstring for presets.

The simulator:
1. Queries a circular footprint from the TileDB array
2. Builds a vertical return histogram (0.15 m bins)
3. Convolves it with a TX pulse kernel (empirical GEDI beam shape, or Gaussian fallback)
4. Detects the ground return
5. Computes cumulative RH metrics from ground up

**Paths** — replace `/path/to/...` with your actual paths.

## Setup

In [ ]:
from alsdb import ALSProvider

reader = ALSProvider(storage_type="local", uri="/path/to/my_array")

## Single footprint simulation

`simulate_waveform` simulates a single large-footprint waveform (25 m diameter by default).  
Centre coordinates must be in the same CRS as the TileDB array (typically UTM).

In [ ]:
from alsdb.processing.waveform import simulate_waveform

result = simulate_waveform(
    provider=reader,
    center_x=308_500.0,
    center_y=4_689_000.0,
    footprint_radius=12.5,  # 25 m diameter, matching GEDI
    year=2021,
)

print(f"RH50  = {result.rh[50]:.2f} m")  # height of median energy
print(f"RH98  = {result.rh[98]:.2f} m")  # equivalent to GEDI rh98
print(f"Cover = {result.cover:.3f}")  # canopy cover fraction
print(f"Ground elevation = {result.z_ground:.2f} m")

## Visualise the waveform

Two plot styles are available:

- `plot_waveform` — normalised energy vs elevation, annotated with RH levels, ground return (brown) and canopy (green) shaded
- `plot_rh_profile` — GEDI L2A canonical representation: RH(p) curve + energy density W(h) with auto-detected layer peaks

In [ ]:
import matplotlib

matplotlib.rcParams["figure.dpi"] = 120

from alsdb.utils.viz import plot_waveform, plot_rh_profile

# Energy vs elevation, annotated with RH levels
fig = plot_waveform(result)
fig.savefig("waveform.png", dpi=150, bbox_inches="tight")

In [ ]:
# GEDI L2A style: RH(p) curve + W(h) energy density
fig = plot_rh_profile(result)
fig.savefig("rh_profile.png", dpi=150, bbox_inches="tight")

## Batch simulation

`simulate_batch` processes a DataFrame of shot centres in parallel.  
Results can be saved to Parquet for later co-location with real GEDI observations.

In [ ]:
import numpy as np
import pandas as pd
from alsdb.processing.waveform import simulate_batch

# Build a grid of GEDI-like shot centres (60 m spacing)
xs, ys = np.meshgrid(
    np.arange(308_100, 309_900, 60),
    np.arange(4_688_500, 4_689_900, 60),
)
shots = pd.DataFrame({"center_x": xs.ravel(), "center_y": ys.ravel()})
print(f"Simulating {len(shots)} footprints...")

results = simulate_batch(
    provider=reader,
    shots=shots,
    year=2021,
    n_workers=4,
    footprint_radius=12.5,
    output_path="shots_2021.parquet",  # optional; omit for in-memory only
)

print(results[["center_x", "center_y", "rh50", "rh98", "cover"]].head())

## 3-D waveform waterfall

`plot_waveforms_3d` shows all shots as a waterfall of RH(p) curves coloured by a summary metric.  
Use `backend="plotly"` for an interactive version suitable for dense grids.

In [ ]:
from alsdb.utils.viz import plot_waveforms_3d

# Static matplotlib figure
fig = plot_waveforms_3d(results, color_by="rh98", backend="matplotlib")
fig.savefig("waveforms_3d.png", dpi=150, bbox_inches="tight")

In [ ]:
# Interactive plotly version (better for dense grids)
# fig = plot_waveforms_3d(results, color_by="cover", backend="plotly")
# fig.show()

## Compare with GEDI L2A

Co-locate simulated shots with real GEDI observations by matching shot coordinates.  
Both datasets use RH metrics at the same footprint scale (~25 m diameter).

In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

# gedi = pd.read_parquet("/path/to/gedi_l2a_region.parquet")

# # Nearest-neighbour match
# tree = cKDTree(gedi[["longitude_lastbin", "latitude_lastbin"]].values)
# _, idx = tree.query(results[["center_x", "center_y"]].values)
# comparison = results.copy()
# comparison["gedi_rh98"] = gedi.iloc[idx]["rh98"].values

# fig, ax = plt.subplots(figsize=(5, 5))
# ax.scatter(comparison["gedi_rh98"], comparison["rh98"], s=5, alpha=0.4)
# ax.plot([0, 50], [0, 50], "k--", lw=1)
# ax.set_xlabel("GEDI RH98 (m)")
# ax.set_ylabel("Simulated RH98 (m)")
# fig.savefig("gedi_vs_simulated.png", dpi=150)